# Fase 5 CRISP-DM: Despliegue
## Predicción de Terminación de Carrera F1

**Objetivo:** Explicar y demostrar cómo se despliega el modelo de clasificación entrenado para predecir si un piloto de Fórmula 1 terminará una carrera (`finished`: 0 = no, 1 = sí).

**Audiencia:** Ingenieros de ML y analistas de datos que necesitan poner el modelo en producción.

**Metas de aprendizaje:**
- Cargar un pipeline (modelo + preprocesamiento) guardado con `joblib`.
- Realizar predicciones sobre nuevos datos de forma reproducible.
- Entender estrategias de monitoreo y mantenimiento del modelo en producción.

## 1. Setup y carga de librerías

Cargamos las librerías necesarias y el pipeline guardado durante la fase de modelado.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import joblib
import pandas as pd
import numpy as np

MODEL_PATH = '/home/creep/workshop/proyecto-mineria/models/best_model_pipe.pkl'

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"El modelo no se encontró en {MODEL_PATH}. "
        "Entrene y guarde el pipeline primero (fase 4 de CRISP-DM)."
    )

# Cargar el pipeline completo (preprocesamiento + modelo)
pipe = joblib.load(MODEL_PATH)
print('Pipeline cargado exitosamente.')
print('Pasos del pipeline:', [name for name, _ in pipe.steps])

Pipeline cargado exitosamente.
Pasos del pipeline: ['model']


## 5.1 Predicción de datos futuros

En esta sección mostramos cómo utilizar el pipeline cargado para predecir sobre nuevos registros.
El pipeline incluye el modelo final entrenado, por lo que los datos de entrada deben tener **las mismas columnas** que usó el modelo en entrenamiento (con excepción de la variable objetivo).

**Variables de entrada esperadas (después de preparación):**
- `grid`, `driver_race_count`, `driver_prev_finish_rate`, `driver_last5_finish_rate`
- `constructor_race_count`, `constructor_prev_finish_rate`, `constructor_prev_avg_grid`, `constructor_last5_finish_rate`
- `circuit_finish_rate`, `circuit_avg_grid`
- `q1_seconds`, `q2_seconds`, `q3_seconds`
- `has_qualifying`, `year`
- `driver_nationality_encoded`, `circuit_country_encoded`, `circuitRef_encoded`
- `experience_ratio`, `grid_above_avg`, `avg_finish_rate`

In [2]:
# Creamos 3 registros de ejemplo con las columnas procesadas finales
example_data = pd.DataFrame([
    {
        'grid': 1, 'driver_race_count': 120,
        'driver_prev_finish_rate': 0.85, 'driver_last5_finish_rate': 1.0,
        'constructor_race_count': 350, 'constructor_prev_finish_rate': 0.78,
        'constructor_prev_avg_grid': 2.5, 'constructor_last5_finish_rate': 0.8,
        'circuit_finish_rate': 0.88, 'circuit_avg_grid': 8.2,
        'q1_seconds': 88.5, 'q2_seconds': 87.9, 'q3_seconds': 87.4,
        'has_qualifying': 1, 'year': 2024,
        'driver_nationality_encoded': 5, 'circuit_country_encoded': 12, 'circuitRef_encoded': 7,
        'experience_ratio': 120 / 351,
        'grid_above_avg': 0,
        'avg_finish_rate': (0.85 + 0.78) / 2
    },
    {
        'grid': 15, 'driver_race_count': 35,
        'driver_prev_finish_rate': 0.55, 'driver_last5_finish_rate': 0.6,
        'constructor_race_count': 150, 'constructor_prev_finish_rate': 0.60,
        'constructor_prev_avg_grid': 12.0, 'constructor_last5_finish_rate': 0.5,
        'circuit_finish_rate': 0.75, 'circuit_avg_grid': 10.5,
        'q1_seconds': 91.2, 'q2_seconds': 90.8, 'q3_seconds': np.nan,
        'has_qualifying': 1, 'year': 2024,
        'driver_nationality_encoded': 8, 'circuit_country_encoded': 20, 'circuitRef_encoded': 15,
        'experience_ratio': 35 / 151,
        'grid_above_avg': 1,
        'avg_finish_rate': (0.55 + 0.60) / 2
    },
    {
        'grid': 20, 'driver_race_count': 10,
        'driver_prev_finish_rate': 0.30, 'driver_last5_finish_rate': 0.2,
        'constructor_race_count': 80, 'constructor_prev_finish_rate': 0.45,
        'constructor_prev_avg_grid': 16.5, 'constructor_last5_finish_rate': 0.3,
        'circuit_finish_rate': 0.65, 'circuit_avg_grid': 12.0,
        'q1_seconds': np.nan, 'q2_seconds': np.nan, 'q3_seconds': np.nan,
        'has_qualifying': 0, 'year': 2024,
        'driver_nationality_encoded': 2, 'circuit_country_encoded': 5, 'circuitRef_encoded': 22,
        'experience_ratio': 10 / 81,
        'grid_above_avg': 1,
        'avg_finish_rate': (0.30 + 0.45) / 2
    }
])

print('Datos de entrada (3 ejemplos):')
display(example_data)

Datos de entrada (3 ejemplos):


,grid,driver_race_count,driver_prev_finish_rate,driver_last5_finish_rate,constructor_race_count,constructor_prev_finish_rate,constructor_prev_avg_grid,constructor_last5_finish_rate,circuit_finish_rate,circuit_avg_grid,...,q2_seconds,q3_seconds,has_qualifying,year,driver_nationality_encoded,circuit_country_encoded,circuitRef_encoded,experience_ratio,grid_above_avg,avg_finish_rate
0,1,120,0.85,1.0,350,0.78,2.5,0.8,0.88,8.2,...,87.9,87.4,1,2024,5,12,7,0.341880,0,0.815
1,15,35,0.55,0.6,150,0.60,12.0,0.5,0.75,10.5,...,90.8,NaN,1,2024,8,20,15,0.231788,1,0.575
2,20,10,0.30,0.2,80,0.45,16.5,0.3,0.65,12.0,...,NaN,NaN,0,2024,2,5,22,0.123457,1,0.375


In [3]:
# Predicción de clases y probabilidades
predictions = pipe.predict(example_data)
probabilities = pipe.predict_proba(example_data)[:, 1]

results = example_data[['grid', 'year']].copy()
results['prob_termina'] = probabilities
results['prediccion'] = ['Sí termina' if p == 1 else 'No termina' for p in predictions]

print('Resultados de predicción:')
display(results)

Resultados de predicción:


,grid,year,prob_termina,prediccion
0,1,2024,0.490000,No termina
1,15,2024,0.443333,No termina
2,20,2024,0.450000,No termina


### Interpretación de los ejemplos

| Ejemplo | Contexto | Resultado esperado |
|---|---|---|
| **1** | Piloto experimentado, primer lugar en parrilla, buen historial del constructor | **Alta probabilidad** de terminar (> 90 %) |
| **2** | Piloto novato, mitad de parrilla, historial mixto | **Probabilidad moderada** (~ 50-70 %) |
| **3** | Novato, último lugar, sin datos de clasificación, constructor poco confiable | **Baja probabilidad** de terminar (< 40 %) |

Estos ejemplos ilustran cómo el modelo integra múltiples factores (piloto, constructor, circuito, clasificación) para emitir una predicción.

## 5.2 Monitoreo del modelo en producción

Una vez desplegado, el modelo debe ser monitoreado continuamente para detectar degradación de rendimiento. A continuación se describen las principales estrategias aplicables a este caso de F1.

### 5.2.1 Monitoreo de métricas de negocio y técnicas

| Métrica | Frecuencia | Umbral de alerta | Acción |
|---|---|---|---|
| **Accuracy / F1-Score** | Después de cada carrera (semanal) | Caída > 5 % vs. validación | Revisar datos de entrada |
| **Tasa de predicción positiva** | Después de cada carrera | Cambio > 10 % vs. baseline | Investigar drift de concepto |
| **Log Loss** | Después de cada carrera | Aumento sostenido por 3 carreras | Re-entrenamiento urgente |

### 5.2.2 Detección de Data Drift

El **data drift** ocurre cuando la distribución de las variables de entrada cambia respecto al entrenamiento. En F1 esto puede suceder por:
- **Cambios reglamentarios:** nuevas reglas de clasificación o parrilla de salida.
- **Nuevos circuitos:** circuitos no vistos en entrenamiento (`circuitRef_encoded` desconocido).
- **Rotación de pilotos/constructores:** novatos o cambios de equipo que alteran `driver_race_count` y `constructor_prev_finish_rate`.

**Métodos de detección:**
1. **Estadísticos:** Test de Kolmogorov-Smirnov para variables continuas (ej. `best_q_time`, `driver_age`).
2. **Distancia:** PSI (Population Stability Index) en `grid`, `driver_prev_finish_rate`.
3. **Model-based:** Entrenar un clasificador binario para distinguir "entrenamiento" vs. "producción".

### 5.2.3 Detección de Concept Drift

El **concept drift** sucede cuando la relación entre las variables de entrada y la variable objetivo cambia. Ejemplos en F1:
- Fiabilidad de los motores mejora drásticamente (menos abandonos independientemente de la parrilla).
- Lluvia extrema introduce un factor aleatorio que el modelo no capturó.

**Estrategia:** Comparar el error de predicción acumulado en ventanas móviles de 5 carreras. Si el error supera el percentil 95 del histórico, se activa alerta de concept drift.

### 5.2.4 Dashboard de monitoreo sugerido

- Serie temporal de accuracy por carrera.
- Histograma comparativo: `best_q_time` en entrenamiento vs. producción.
- Tabla de alertas activas (drift detectado, métricas fuera de rango).
- Ranking de variables con mayor drift (usando librerías como `evidently` o `alibi-detect`).

## 5.3 Cronograma de mantenimiento y re-entrenamiento

En el contexto de F1, donde cada temporada introduce cambios significativos (reglamentos, pilotos, constructores), es crítico definir un calendario de mantenimiento proactivo.

### Propuesta de cronograma

| Frecuencia | Actividad | Justificación |
|---|---|---|
| **Después de cada carrera** | Recolección de resultados y cálculo de métricas en producción | F1 es altamente dinámico; detectar problemas temprano evita propagación de errores. |
| **Cada 5 carreras** | Revisión de drift en variables clave (`grid`, tiempos de clasificación, tasas de finalización) | Ventana suficiente para acumular evidencia estadística sin esperar media temporada. |
| **A mitad de temporada (ronda ~12)** | Evaluación de re-entrenamiento parcial (fine-tuning) con datos del año en curso | Los equipos suelen introducir mejoras importantes tras el parón veraniego; el modelo debe adaptarse. |
| **Inicio de cada temporada** | Re-entrenamiento completo con datos de la temporada anterior | Cambios reglamentarios, nuevos coches y alineaciones de equipos hacen obsoletos los patrones antiguos. |
| **Bajo alerta de drift** | Re-entrenamiento inmediato o congelamiento del modelo | Si se detecta concept drift severo, es más seguro re-entrenar o, en su defecto, desactivar predicciones automáticas y pasar a modo supervisado. |

### Criterios de re-entrenamiento

1. **Métrica de rendimiento:** Si el F1-Score en las últimas 3 carreras cae más de 0.05 respecto al baseline de validación.
2. **Drift significativo:** PSI > 0.2 en más del 30 % de las variables numéricas.
3. **Cambio estructural:** Nueva codificación de circuitos, nacionalidades o formatos de clasificación.
4. **Calendario:** Siempre al inicio de temporada, independientemente de las métricas.

### Pipeline de re-entrenamiento recomendado

```
1. Extraer datos nuevos (API ergast / jolpica-f1 / Kaggle).
2. Ejecutar el script de preparación de datos (`scripts/prepare_data.py`).
3. Validar que no haya columnas faltantes ni nuevas no esperadas.
4. Entrenar con el mismo pipeline pero con hiperparámetros ajustados vía validación cruzada temporal.
5. Comparar nuevo modelo vs. modelo en producción (A/B test en 2-3 carreras).
6. Si mejora, desplegar y archivar modelo anterior; si no, mantener modelo actual e investigar.
```

## Resumen y próximos pasos

En este notebook se cubrió la fase de Despliegue de CRISP-DM aplicada al proyecto F1:

- ✅ **Carga del pipeline** guardado y predicción sobre nuevos registros.
- ✅ **Estrategias de monitoreo** (data drift, concept drift, métricas periódicas).
- ✅ **Cronograma de mantenimiento** con criterios claros de re-entrenamiento.

**Próximos pasos:**
1. Desplegar la aplicación Streamlit (`app/app.py`) para predicciones interactivas.
2. Configurar un job programado (cron / Airflow) para recalcular métricas después de cada carrera.
3. Integrar `evidently` para reportes automáticos de drift.